In [2]:
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
from src.helper import download_hugging_face_embeddings

print("Opening fresh connection to your local database folder...")
embeddings = download_hugging_face_embeddings()
client = QdrantClient(path="../qdrant_db")

db = QdrantVectorStore(
    client=client, 
    collection_name="medical_chatbot", 
    embedding=embeddings
)

# Test query to check if semantic search reads accurately
query = "What are the early symptoms of Asthma?"
docs = db.similarity_search(query, k=2)

print("\n--- Similarity Search Results ---")
for idx, doc in enumerate(docs):
    print(f"\n[Matched Chunk {idx+1}]:\n{doc.page_content}")
    print("-" * 40)

# Always close the local file stream handle!
client.close()

Opening fresh connection to your local database folder...


C:\Users\sweta\AppData\Local\Temp\ipykernel_25736\3646525780.py:7: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <medical_chatbot> contains 40135 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client = QdrantClient(path="../qdrant_db")



--- Similarity Search Results ---

[Matched Chunk 1]:
Causes and symptoms
In most cases, asthma is caused by inhaling an
allergen that sets off the chain of biochemical and
tissue
changes
leading
to
airway
inflammation,
bronchoconstriction, and wheezing. Because avoiding
(or at least minimizing) exposure is the most effective
way of treating asthma, it is vital to identify which
allergen or irritant is causing symptoms in a particular
patient. Once asthma is present, symptoms can be set
off or made worse if the patient also has rhinitis
----------------------------------------

[Matched Chunk 2]:
a very sensitive person to develop symptoms of occu-
pational asthma. A person who has occupational
asthma has one or more symptoms, including cough-
ing, shortness of breath, tightness in the chest, and
wheezing. Symptoms may appear less than 24 hours
after the person is first exposed to the irritant or
develop two or three years later.
At first, symptoms appear while the person is at
work o

In [1]:
import sys
import os
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from src.helper import download_hugging_face_embeddings, load_pdf_data, text_split

# 1. Setup paths and load data
base_dir = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
data_folder_path = os.path.join(base_dir, "data")
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

print("Loading text chunks and embedding model into memory...")
extracted_data = load_pdf_data(data_folder_path)
text_chunks = text_split(extracted_data)
embeddings = download_hugging_face_embeddings()

collection_name = "medical_chatbot"
db_path = "../qdrant_db"

# 2. Open ONE single explicit client connection
print("\nStep 1: Connecting to local Qdrant engine...")
client = QdrantClient(path=db_path)

print("Recreating collection space...")
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

# 3. Create the store by passing the client object ONLY (Do NOT pass path here!)
print("\nStep 2: Vectorizing and storing chunks (Running local batch loops)...")
print("⏳ This will take 5 to 15 minutes because it processes 40,135 elements. Please wait...")

vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings
)

# Add the documents safely through the single active connection
vector_store.add_documents(documents=text_chunks)

# 4. Explicitly close the file handle lock when finished
client.close()
print("\n🎉 Success! All 40,135 text chunks have been successfully saved inside your local 'qdrant_db' folder.")

d:\Medical_chatbot\Medical-Chatbot-using-RAG-LLM-Langchain-Pinecone-Flask-and-AWS\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading text chunks and embedding model into memory...
Targeting file directly: d:\Medical_chatbot\data\medical_book.pdf

Step 1: Connecting to local Qdrant engine...
Recreating collection space...

Step 2: Vectorizing and storing chunks (Running local batch loops)...
⏳ This will take 5 to 15 minutes because it processes 40,135 elements. Please wait...


C:\Users\sweta\AppData\Local\Temp\ipykernel_25736\1907861136.py:26: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(
d:\Medical_chatbot\Medical-Chatbot-using-RAG-LLM-Langchain-Pinecone-Flask-and-AWS\venv\Lib\site-packages\langchain_qdrant\qdrant.py:483: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20032 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  self.client.upsert(



🎉 Success! All 40,135 text chunks have been successfully saved inside your local 'qdrant_db' folder.
